# Extract extra data from the raw Lending Club file (run once, before deleting it)
Reads the Kaggle `accepted_2007_to_2018Q4` file in `Desktop/Q2/archive` and writes, to `Desktop/Q2/extra_data/`:

| file | contents |
|---|---|
| `lending_club_36m_wide.csv` | resolved **36-month** loans issued <= 2015-12: the 15 columns the notebook uses **plus** extra columns (`grade`, `sub_grade`, `installment`, ...) |
| `lending_club_60m_wide.csv` | resolved **60-month** loans issued <= 2013-12 (the latest vintage that has fully resolved by the 2018-Q4 extract), same columns |
| `raw_manifest.json` | name, size and **SHA-256** of the raw file, row counts, filters used, date of extraction |

Choose **Run All**. Nothing is modelled or simulated; every row is a real loan. A few minutes, a few GB of RAM at most.

> Columns such as `total_pymnt`, `recoveries` or `last_pymnt_d` are only known *after* a loan ends, so they must **never** be used as model inputs; they are not extracted.


In [ ]:
import os, io, gzip, json, hashlib, zipfile, datetime
from pathlib import Path
import numpy as np, pandas as pd

def find_q2():
    if os.environ.get('Q2_DIR'): return Path(os.environ['Q2_DIR'])
    home = Path.home()
    for d in [home/'Desktop', home/'OneDrive'/'Desktop', *sorted(home.glob('OneDrive*/Desktop'))]:
        if (d/'Q2').is_dir(): return d/'Q2'
Q2 = find_q2()
assert Q2 is not None, "No 'Q2' folder on your Desktop. Set Q2 = Path(r'...your path...') in this cell."
ARCHIVE, OUT = Q2/'archive', Q2/'extra_data'
OUT.mkdir(exist_ok=True)
print("Q2:", Q2, "| output:", OUT)

def find_raw():
    files = [p for p in ARCHIVE.rglob('*') if p.is_file()] if ARCHIVE.is_dir() else []
    acc = [p for p in files if 'accepted' in p.name.lower() and p.name.lower().endswith(('.csv', '.csv.gz'))]
    if acc: return max(acc, key=lambda p: p.stat().st_size)
    for z in list(Q2.glob('*.zip')) + [p for p in files if p.suffix.lower() == '.zip']:
        with zipfile.ZipFile(z) as zf:
            hit = [n for n in zf.namelist() if 'accepted' in n.lower() and n.lower().endswith(('.csv', '.csv.gz'))]
            if hit: return (z, hit[0])
    raise FileNotFoundError(f"No accepted-loans file under {ARCHIVE}. Found: {[str(p.relative_to(Q2)) for p in files[:20]]}")

def open_raw(raw):
    if isinstance(raw, tuple):
        zf = zipfile.ZipFile(raw[0]); f = zf.open(raw[1])
        return gzip.open(f) if raw[1].lower().endswith('.gz') else f
    return raw

def sha256_of(path, block=1 << 24):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(block), b''): h.update(b)
    return h.hexdigest()

RAW = find_raw()
RAW_FILE = RAW[0] if isinstance(RAW, tuple) else RAW           # the file that actually sits on disk (zip, gz or csv)
print("Raw file:", RAW if not isinstance(RAW, tuple) else f"{RAW[0]} -> {RAW[1]}")
print("Hashing the raw file on disk (about a minute)...")
RAW_SHA, RAW_SIZE = sha256_of(RAW_FILE), RAW_FILE.stat().st_size
print("SHA-256:", RAW_SHA, "| bytes:", f"{RAW_SIZE:,}")

In [ ]:
BASE = ['loan_amnt', 'term', 'int_rate', 'annual_inc', 'dti', 'fico_range_low', 'fico_range_high', 'revol_util', 'delinq_2yrs',
        'emp_length', 'home_ownership', 'verification_status', 'purpose', 'issue_d', 'loan_status']
# Known at origination (application / credit-bureau data at issue time). Used only if present in the file.
EXTRA = ['id', 'grade', 'sub_grade', 'installment', 'funded_amnt', 'open_acc', 'pub_rec', 'total_acc', 'earliest_cr_line',
         'inq_last_6mths', 'mths_since_last_delinq', 'mort_acc', 'pub_rec_bankruptcies', 'revol_bal', 'addr_state',
         'application_type', 'initial_list_status', 'zip_code']
GOOD, BAD = {'Fully Paid'}, {'Charged Off', 'Default'}
head = pd.read_csv(open_raw(RAW), nrows=0).columns
missing_base = [c for c in BASE if c not in head]
assert not missing_base, f"Raw file lacks required columns {missing_base}: this needs the 'accepted' file."
extra_present = [c for c in EXTRA if c in head]
print("Extra columns found:", extra_present)
print("Extra columns NOT in this file:", [c for c in EXTRA if c not in head])

parts, n_raw = [], 0
for ch in pd.read_csv(open_raw(RAW), usecols=BASE + extra_present, chunksize=250_000, low_memory=False):
    n_raw += len(ch)
    ch = ch[ch['loan_status'].isin(GOOD | BAD)].copy()
    ch['term'] = pd.to_numeric(ch['term'].astype(str).str.extract(r'(\d+)')[0], errors='coerce')
    parts.append(ch[ch['term'].isin([36, 60])])
D = pd.concat(parts, ignore_index=True)
print(f"Raw rows read: {n_raw:,} | resolved 36/60-month loans: {len(D):,}")

D['issue_d'] = pd.to_datetime(D['issue_d'].astype(str).str.strip(), format='%b-%Y', errors='coerce')
D = D.dropna(subset=['issue_d'])
for c in ('int_rate', 'revol_util'):
    D[c] = pd.to_numeric(D[c].astype(str).str.rstrip('%'), errors='coerce')
D['fico_score'] = (D['fico_range_low'] + D['fico_range_high']) / 2
e = D['emp_length'].astype('string').str.strip()
D['emp_length_yrs'] = e.str.extract(r'(\d+)')[0].astype('float'); D.loc[e.str.startswith('<', na=False), 'emp_length_yrs'] = 0.0
D['default'] = D['loan_status'].isin(BAD).astype(int)
D['year'], D['month'] = D['issue_d'].dt.year, D['issue_d'].dt.month
if 'earliest_cr_line' in D:                                    # credit history length in years at issue (a usable origination feature)
    D['credit_hist_yrs'] = (D['issue_d'] - pd.to_datetime(D['earliest_cr_line'], format='%b-%Y', errors='coerce')).dt.days / 365.25
CORE = ['year', 'month', 'issue_d', 'default', 'loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_score', 'revol_util',
        'delinq_2yrs', 'emp_length_yrs', 'home_ownership', 'verification_status', 'purpose']
EXTRA_OUT = [c for c in extra_present if c != 'earliest_cr_line'] + (['credit_hist_yrs'] if 'credit_hist_yrs' in D else [])
COLS = CORE + ['term', 'loan_status'] + EXTRA_OUT

In [ ]:
def save(term, max_issue, name):
    d = D[(D['term'] == term) & (D['issue_d'] <= pd.Period(max_issue).to_timestamp(how='end'))]
    n0 = len(d); d = d.dropna(subset=CORE)                       # extras may stay missing; the 15 core columns may not
    d = d.sort_values('issue_d', kind='stable')[COLS].copy(); d['issue_d'] = d['issue_d'].dt.strftime('%Y-%m-%d')
    path = OUT / name; d.to_csv(path, index=False)
    print(f"\n{name}: {len(d):,} rows ({n0 - len(d):,} dropped for missing core columns) | issue <= {max_issue} | default rate {d['default'].mean():.2%}")
    print(d.groupby('year')['default'].agg(N='size', default_rate='mean').round(4).T.to_string())
    return dict(file=name, term=term, max_issue=max_issue, rows=int(len(d)), default_rate=float(d['default'].mean()),
                years=[int(d['year'].min()), int(d['year'].max())], sha256=sha256_of(path))
info36 = save(36, '2015-12', 'lending_club_36m_wide.csv')
info60 = save(60, '2013-12', 'lending_club_60m_wide.csv')

In [ ]:
manifest = dict(created=datetime.datetime.now().isoformat(timespec='seconds'), raw_file=RAW_FILE.name, raw_bytes=RAW_SIZE, raw_sha256=RAW_SHA,
                raw_rows_read=int(n_raw), source="Kaggle 'All Lending Club loan data' (Nathan George), accepted_2007_to_2018Q4, CC0",
                filters="loan_status in {Fully Paid, Charged Off, Default}; 36-month issued <= 2015-12; 60-month issued <= 2013-12 "
                        "(so every loan had its full term to resolve before the 2018-Q4 extract); core columns complete",
                extra_columns=EXTRA_OUT, outputs=[info36, info60])
(OUT / 'raw_manifest.json').write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
print("\nDone. Back up the whole 'extra_data' folder AND the raw zip before deleting anything.")